In [1]:
import sys
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "results/assemblyworldbench/benchmark/benchmark.json").exists()
)
RESULTS = ROOT / "results"
PAPER = ROOT.parent / "AssemblyWorldBench"
CACHE = ROOT / "notebooks/.cache/paper-analysis"
RECOMPUTE = False
WORKERS = 6
COMPUTE_MISSING = True  # Score missing recorded states; no agent is run.
sys.path.insert(0, str(ROOT / "notebooks"))
import paper_analysis as analysis  # noqa: E402

analysis.configure(RESULTS, PAPER, CACHE)

COMPUTE_MISSING = True  # Set True to score missing recorded states; no agent is run.

# Recorded behavior and geometric progress

All experiment inputs come from the exported `results/` package. No historical run directories or temporary scratchpad are read. The geometry analysis uses the existing keyed evaluator-input interface to resolve dataset-side point clouds and target poses. Derived statistics and caches remain in `notebooks/.cache/paper-analysis/`. PDF figures are written to the sibling paper's `fig/` directories. Run with the project environment (`uv sync --extra episodes --group inspection`).

Intervals are pointwise shape-paired, source-stratified bootstrap intervals; they are not simultaneous significance claims. Missing input records are reported explicitly, never replaced with numbers from a report.

In [2]:
import paper_figures as figures
import paper_geometry as geometry

frame = analysis.load_benchmark()
timing_audit = geometry.audit_timing(frame)
display(timing_audit)

/Users/davidz/Projects/MERL/AssemblyWorldModel/assembly-world-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'episodes': 800,
 'calls_with_completion_event': 116492,
 'max_request_completion_seconds': 7.908,
 'calls_over_one_second': 165,
 'changed': [{'system': 'gpt-6-astra',
   'block': 'fantastic-breaks-none',
   'sample_id': '02/02007',
   'before': ('fraction', 0.8, 47),
   'after': ('fraction', 0.8, 49)},
  {'system': 'claude-sonnet-5',
   'block': 'assemblybench-manualbook',
   'sample_id': '7780',
   'before': ('fraction', 0.2, 11),
   'after': ('fraction', 0.2, 12)},
  {'system': 'gpt-5.6-terra',
   'block': 'partnet-none',
   'sample_id': '45594',
   'before': ('fraction', 0.5, 3),
   'after': ('fraction', 0.5, 6)}],
 'timestamp_basis': 'environment completion events',
 'post_60_minute_pose_changes': [{'system': 'claude-opus-5',
   'block': 'assemblybench-manualbook',
   'sample_id': '1386',
   'at60': 129,
   'final': 130},
  {'system': 'qwen3.8-max-litellm',
   'block': 'partnet-final-image',
   'sample_id': '23108',
   'at60': 56,
   'final': 60},
  {'system': 'qwen3.8-max-litel

## Exact recorded-state evaluation
The scorer evaluates the state after the last completed operation at each absolute budget and each decile of the first-to-last completed environment-call interval. Identical poses share a validated cache. Final scores are independently recomputed at the recorded official alignment. Additional intermediate states use the unchanged multistart registration and Hungarian scorer. This is retrospective truncation, not a new budget-conditioned run.

Computing missing geometry scores can take several hours. Six worker processes limit concurrency; the default renders only validated existing caches and reports incomplete coverage.

In [3]:
if COMPUTE_MISSING:
    validation = geometry.compute_trajectories(frame, workers=WORKERS, recompute=RECOMPUTE)
    display(validation.groupby(["system", "ok"]).size())
analyses, missing = figures.load_analyses(frame)
if missing:
    import paper_geometry as geometry
    geometry.compute_trajectories(frame, workers=WORKERS, score=False)
    analyses, missing = figures.load_analyses(frame)
print("Episodes with validated geometry scores:", sum(bool(d["scores"]) for d in analyses), "/", len(frame))
print("Available episode analyses:", len(analyses), "Missing:", missing)

10/800 episodes; last={'system': 'gpt-6-astra', 'block': 'partnet-none', 'sample_id': '37433', 'ok': True, 'states': 9, 'seconds': 0.1803695830021752}


20/800 episodes; last={'system': 'gpt-6-astra', 'block': 'partnet-none', 'sample_id': '743', 'ok': True, 'states': 11, 'seconds': 0.12361025000427617}


30/800 episodes; last={'system': 'gpt-6-astra', 'block': 'partnet-final-image', 'sample_id': '37433', 'ok': True, 'states': 12, 'seconds': 0.1577576659983606}


40/800 episodes; last={'system': 'gpt-6-astra', 'block': 'partnet-final-image', 'sample_id': '743', 'ok': True, 'states': 11, 'seconds': 0.12695679200260201}


50/800 episodes; last={'system': 'gpt-6-astra', 'block': 'ikea-manualbook', 'sample_id': 'Chair/reidar', 'ok': True, 'states': 10, 'seconds': 0.09517666600004304}


...

780/800 episodes; last={'system': 'deepseek-v4.1-flash', 'block': 'assemblybench-manualbook', 'sample_id': '8243', 'ok': True, 'states': 1, 'seconds': 0.09992016599426279}


790/800 episodes; last={'system': 'deepseek-v4.1-flash', 'block': 'fantastic-breaks-none', 'sample_id': '03/03000', 'ok': True, 'states': 8, 'seconds': 0.1403035830007866}


800/800 episodes; last={'system': 'deepseek-v4.1-flash', 'block': 'fantastic-breaks-none', 'sample_id': '19/19003', 'ok': True, 'states': 10, 'seconds': 0.11995129199931398}


system               ok  
claude-fable-5-1     True    100
claude-opus-5        True    100
claude-sonnet-5      True    100
deepseek-v4.1-flash  True    100
gpt-5.6-sol          True    100
gpt-5.6-terra        True    100
gpt-6-astra          True    100
qwen3.8-max-litellm  True    100
dtype: int64

Episodes with validated geometry scores: 800 / 800
Available episode analyses: 800 Missing: []


In [4]:
actions = figures.make_behavior(frame, analyses)
quality = figures.make_quality(frame, analyses)
figures.make_main_behavior(analyses, quality)
strategy = figures.make_strategy(frame, analyses)
display(strategy.groupby("system").median(numeric_only=True))
display(quality)

,edit_count,capture_count,revisit_fraction,reversal_fraction,captures_per_edit,final_camera_ratio
system,,,,,,
claude-fable-5-1,49.0,40.0,0.875000,0.602083,0.805556,0.334013
claude-opus-5,50.0,57.0,0.886335,0.685213,1.121345,0.334134
claude-sonnet-5,61.0,54.5,0.898718,0.686738,0.954286,0.680400
deepseek-v4.1-flash,52.5,13.0,0.927536,0.697716,0.230769,0.861787
gpt-5.6-sol,63.0,39.5,0.892582,0.620050,0.641053,0.552799
gpt-5.6-terra,19.5,12.0,0.722222,0.625000,0.625000,1.000000
gpt-6-astra,32.0,23.0,0.816901,0.566964,0.699095,0.339385
qwen3.8-max-litellm,49.0,23.0,0.875000,0.600000,0.413665,0.828578


,system,kind,time,PA,SR,SCD_median,PA_ci
0,claude-fable-5-1,budget,1.0,0.001563,0.00000,444.269876,"[0.0, 0.0046875]"
1,claude-fable-5-1,budget,2.0,0.014063,0.01250,516.792320,"[0.0, 0.040625]"
2,claude-fable-5-1,budget,5.0,0.143934,0.08750,283.360938,"[0.08571428571428572, 0.20768574134199133]"
3,claude-fable-5-1,budget,10.0,0.317669,0.21875,37.668169,"[0.23221559690309693, 0.4077526848151848]"
4,claude-fable-5-1,budget,20.0,0.614058,0.41875,1.719576,"[0.5324437966547342, 0.6920241763444889]"
...,...,...,...,...,...,...,...
147,qwen3.8-max-litellm,fraction,0.6,0.043611,0.02500,722.058256,"[0.01111111111111111, 0.08611111111111111]"
148,qwen3.8-max-litellm,fraction,0.7,0.044306,0.02500,639.443741,"[0.01111111111111111, 0.08680555555555555]"
149,qwen3.8-max-litellm,fraction,0.8,0.114242,0.09375,418.770352,"[0.05965909090909092, 0.17522727272727273]"
150,qwen3.8-max-litellm,fraction,0.9,0.146944,0.11875,580.823493,"[0.09, 0.20944444444444446]"


## Coverage and endpoint validation
Curves are emitted only for systems with all 100 episodes scored. Never infer a thinking duration from gaps between tool calls. PA regressions may include changes in the shared registration or equivalent-part assignment.

In [5]:
if (CACHE / "trajectory_validation.json").exists():
    display(analysis.read_json(CACHE / "trajectory_validation.json")[-10:])
regressions = geometry.audit_regressions(frame, analyses)
if not regressions.empty:
    display(regressions.groupby("system")[
        ["drop_persists_frozen_alignment", "drop_persists_frozen_alignment_and_matching", "matching_changes"]
    ].agg(["count", "sum"]))


[{'system': 'deepseek-v4.1-flash',
  'block': 'fantastic-breaks-none',
  'sample_id': '05/05005',
  'ok': True,
  'states': 8,
  'seconds': 0.07152208399929805},
 {'system': 'deepseek-v4.1-flash',
  'block': 'fantastic-breaks-none',
  'sample_id': '07/07004',
  'ok': True,
  'states': 9,
  'seconds': 0.1464912919982453},
 {'system': 'deepseek-v4.1-flash',
  'block': 'fantastic-breaks-none',
  'sample_id': '09/09000',
  'ok': True,
  'states': 5,
  'seconds': 0.11284316700039199},
 {'system': 'deepseek-v4.1-flash',
  'block': 'fantastic-breaks-none',
  'sample_id': '09/09022',
  'ok': True,
  'states': 13,
  'seconds': 0.06017579199397005},
 {'system': 'deepseek-v4.1-flash',
  'block': 'fantastic-breaks-none',
  'sample_id': '09/09028',
  'ok': True,
  'states': 9,
  'seconds': 0.13853408300201409},
 {'system': 'deepseek-v4.1-flash',
  'block': 'fantastic-breaks-none',
  'sample_id': '09/09029',
  'ok': True,
  'states': 8,
  'seconds': 0.1351320419998956},
 {'system': 'deepseek-v4.1-fl

drop_persists_frozen_alignment      \
                                             count sum   
system                                                   
claude-fable-5-1                                61  30   
claude-opus-5                                   50  31   
claude-sonnet-5                                 20  12   
deepseek-v4.1-flash                              7   7   
gpt-5.6-sol                                     37  21   
gpt-5.6-terra                                   19   8   
gpt-6-astra                                     28  22   
qwen3.8-max-litellm                             17   9   

                    drop_persists_frozen_alignment_and_matching      \
                                                          count sum   
system                                                                
claude-fable-5-1                                             61  31   
claude-opus-5                                                50  29   
claude-sonnet-5                                              20  12   
deepseek-v4.1-flash                                           7   7   
gpt-5.6-sol                                                  37  20   
gpt-5.6-terra                                                19   8   
gpt-6-astra                                                  28  21   
qwen3.8-max-litellm                                          17   9   

                    matching_changes       
                               count  sum  
system                                     
claude-fable-5-1                  61  190  
claude-opus-5                     50  104  
claude-sonnet-5                   20   20  
deepseek-v4.1-flash                7   12  
gpt-5.6-sol                       37  113  
gpt-5.6-terra                     19   11  
gpt-6-astra                       28   46  
qwen3.8-max-litellm               17   15

## Optional remote CPU acceleration
`paper_remote_score.py` can evaluate exact states on a user-controlled SSH host. It stages only the scorer source; evaluator arrays are streamed in memory and scores return to this local notebook cache. Remote NumPy/SciPy versions are recorded. Run `compute_trajectories` afterward to assemble the per-episode cache and validate final scores. The default workflow does not contact a remote host.
